In [0]:
%run ./silver_utils

# Silver utilities

Reusable class for cleaning bronze DataFrames into silver tables.
Load it in another notebook with:

```
%run ./silver_utils
```

## Example: silver orders

Uncomment to run here, or copy into your own notebook after `%run ./silver_utils`.

In [0]:
from pyspark.sql import functions as F

(SilverTransformer.from_table("databricks_catalog.bronze_schema.regions")
    .trim()                                       # remove spaces from all text columns
    .drop_nulls("region_id", "region")            # a region needs an ID and a name
    .deduplicate("region_id")                     # one row per region
    .apply(lambda df: df
           .withColumn("region_id", F.upper("region_id"))     # "r01" → "R01"
           .withColumn("region",    F.initcap("region")))     # "EAST" → "East"
    .add_processed_at()
    .drop("_rescued_data")
    .write("databricks_catalog.silver_schema.regions"))

✓ Written to databricks_catalog.silver_schema.regions


DataFrame[region_id: string, region: string, processed_at: timestamp]